[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/1-using-tool.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab: clone the repo and install packages. Local Jupyter skips this.
# Set OPENAI_API_KEY in Colab secrets, or in the repo-root .env locally.

import os
import sys
from pathlib import Path


def in_colab():
    """True on Google Colab. False in local Jupyter and VS Code."""
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


if not in_colab():
    print("Not Colab. Skipped clone and pip install.")
else:
    import shutil
    import subprocess

    missing = []
    for module_name, requirement in (
        ("openai", "openai>=1.40.0"),
        ("dotenv", "python-dotenv>=1.0.1"),
        ("httpx", "httpx>=0.27.0"),
    ):
        try:
            __import__(module_name)
        except ImportError:
            missing.append(requirement)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    repo = Path("/content/book-agents")
    marker = repo / "tutorial" / "common" / "client.py"
    if not marker.is_file():
        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not marker.is_file():
            raise RuntimeError(
                "Colab setup could not find tutorial/common/client.py after cloning."
            )

    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        try:
            from google.colab import userdata
            secret = userdata.get("OPENAI_API_KEY")
        except Exception:
            secret = ""
        if secret and str(secret).strip():
            os.environ["OPENAI_API_KEY"] = str(secret).strip()

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError(
            "OPENAI_API_KEY is empty. On Colab, add a secret named OPENAI_API_KEY "
            "(the key icon) and rerun this cell. Locally, copy .env.example to .env "
            "and paste a key from https://platform.openai.com/api-keys. Never commit .env."
        )
    print("Colab: tutorial/ is ready. OPENAI_API_KEY is set (value hidden).")


# 1. Tool calling

One tool, two model calls. The model asks for `get_shop_fact`. This notebook reads the Returns section of `docs/policy.md` and sends that text back. The second call writes the answer from that text.

You need `OPENAI_API_KEY` in the repository-root `.env`, or in a Colab secret of the same name.


## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.


In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import chat, require_key

_, model = require_key()
print("model:", model)


## Test chat

One Chat Completions call, with no tools. This checks the key and the model before the tool run.


In [ ]:
def test_chat(question):
    """Ask one question with no tools, and print the answer."""
    turn = chat([{"role": "user", "content": question}], tools=[])
    print("ANSWER:", turn["content"])


test_chat("Reply with exactly: ok")


## One tool

`get_shop_fact` opens a shop document and returns one section. The model only sees that string. The schema below is what you send to the API.


In [ ]:
def get_shop_fact(args):
    """Open one shop document and return the section for this topic."""
    topic = str(args.get("topic", "")).strip().lower()
    pages = {
        "returns": ("policy.md", "Returns"),
        "shipping": ("policy.md", "Shipping"),
        "hours": ("faq.md", "Where and when"),
        "allergens": ("faq.md", "Allergens"),
    }
    if topic not in pages:
        return "topic must be returns, shipping, hours, or allergens."
    filename, heading = pages[topic]
    lines = (ROOT / "tutorial" / "docs" / filename).read_text().splitlines()
    start = lines.index("## " + heading) + 1
    body = []
    for line in lines[start:]:
        if line.startswith("## "):
            break
        body.append(line)
    text = "\n".join(body).strip()
    return "TOPIC: " + topic + "\nSOURCE: docs/" + filename + "\n\n" + text


tools = [{
    "type": "function",
    "function": {
        "name": "get_shop_fact",
        "description": "Look up one shop rule. topic is returns, shipping, hours, or allergens.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "returns, shipping, hours, or allergens"},
            },
            "required": ["topic"],
        },
    },
}]


## Run

The first call should request `get_shop_fact`. We run that function, append the result, and call the model again.


In [ ]:
question = "Can a customer return an opened bag of house coffee?"
messages = [
    {
        "role": "system",
        "content": (
            "You are the counter concierge for Hearth Lane Café. "
            "Use get_shop_fact before you state a return, shipping, hours, or allergen rule. "
            "Answer from that tool result."
        ),
    },
    {"role": "user", "content": question},
]
print("QUESTION:", question)

first = chat(messages, tools)
calls = first["tool_calls"]
print("tool_calls:", [call["name"] for call in calls])
call = calls[0]

messages.append({
    "role": "assistant",
    "content": first["content"] or "",
    "tool_calls": [{
        "id": call["id"],
        "type": "function",
        "function": {"name": call["name"], "arguments": call["raw"]},
    }],
})
result = get_shop_fact(call["arguments"])
print(result)
messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})

second = chat(messages, tools)
print("ANSWER:", second["content"])
